In [1]:
import collections
import csv
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.set_default_dtype(torch.float64)

# ─── 1. Constants & Vocabulary ───────────────────────────────────────────────
N_PRETERM = 26
START_SYMBOL = "S"
PRETERM_NAMES = [chr(ord('A') + i) for i in range(N_PRETERM)]
# CRUCIAL FIX: Avoid exporting two different symbols named "S"
PRETERM_NAMES[ord('s') - ord('a')] = "PT_S"
TERM_TO_IDX = {chr(ord('a') + i): i for i in range(N_PRETERM)}

def sym_name(idx):
    if idx < N_PRETERM:
        return PRETERM_NAMES[idx]
    if idx == N_PRETERM:
        return START_SYMBOL
    return f"NT{idx - N_PRETERM}"

In [2]:
# ─── 2. Data Loader ──────────────────────────────────────────────────────────
def load_corpus(path, device='cpu'):
    """Loads text corpus and converts strings into PyTorch tensors."""
    sents = []
    with open(path) as f:
        for line in f:
            toks = line.strip().split()
            if toks:
                indices = [TERM_TO_IDX[t] for t in toks]
                sents.append(torch.tensor(indices, dtype=torch.long, device=device))
    return sents

In [3]:
# ─── 3. TD-PCFG PyTorch Module ───────────────────────────────────────────────
class TD_PCFG(nn.Module):
    def __init__(self, n_nt, rank=64):
        super().__init__()
        self.n_nt = n_nt
        self.N = N_PRETERM + n_nt  # Total symbols: 26 preterminals + n_nt nonterminals
        self.START_IDX = N_PRETERM # Index 26 is 'S'
        
        # CP Decomposition parameters (Unnormalized)
        # We initialize with a slight variance to break symmetry
        self.U = nn.Parameter(torch.randn(n_nt, rank) * 0.1)
        self.V = nn.Parameter(torch.randn(self.N, rank) * 0.1)
        self.W = nn.Parameter(torch.randn(self.N, rank) * 0.1)

    def get_dense_rules(self):
        """Reconstructs the full (n_nt, N, N) binary rule probability tensor."""
        u_prob = F.softmax(self.U, dim=1) 
        v_prob = F.softmax(self.V, dim=0) 
        w_prob = F.softmax(self.W, dim=0) 
        
        # T_{a,b,c} = sum_r U_{a,r} * V_{b,r} * W_{c,r}
        return torch.einsum('ar,br,cr->abc', u_prob, v_prob, w_prob)

In [4]:
# ─── 4. Differentiable Inside Algorithm ──────────────────────────────────────
def _inside_probability(rules, sentence_tensor, start_idx):
    n = len(sentence_tensor)
    if n < 2:
        return torch.zeros((), dtype=rules.dtype, device=rules.device)

    alpha = [[None for _ in range(n)] for _ in range(n)]
    n_symbols = rules.shape[1]

    for i in range(n):
        node = torch.zeros(n_symbols, dtype=rules.dtype, device=rules.device)
        node[int(sentence_tensor[i].item())] = 1.0
        alpha[i][i] = node

    for width in range(2, n + 1):
        for i in range(n - width + 1):
            j = i + width - 1
            span_nt = torch.zeros(rules.shape[0], dtype=rules.dtype, device=rules.device)

            for k in range(i, j):
                lv = alpha[i][k]
                rv = alpha[k + 1][j]
                span_nt = span_nt + torch.einsum('abc,b,c->a', rules, lv, rv)

            span_prob = torch.zeros(n_symbols, dtype=rules.dtype, device=rules.device)
            span_prob[N_PRETERM:] = span_nt
            alpha[i][j] = span_prob

    return alpha[0][n - 1][start_idx]


def compute_inside_loss(model, sentence_tensor):
    """Negative log-likelihood with a numerically safer inside pass."""
    p_sent = _inside_probability(model.get_dense_rules(), sentence_tensor, model.START_IDX)
    eps = torch.finfo(p_sent.dtype).tiny
    return -torch.log(p_sent.clamp_min(eps))

In [5]:
def count_active_rules(model, threshold=1e-3):
    with torch.no_grad():
        rules = model.get_dense_rules().detach()
        return int((rules > threshold).sum().item())


def compute_sentence_probability(model, sentence_tensor):
    """Computes sentence probability with the same inside pass used for training."""
    if len(sentence_tensor) < 2:
        return 0.0

    with torch.no_grad():
        rules = model.get_dense_rules().detach()
        sentence_tensor = sentence_tensor.to(rules.device)
        return float(_inside_probability(rules, sentence_tensor, model.START_IDX).item())


def train_model(
    model,
    train_corpus,
    epochs=6,
    lr=0.02,
    max_len=40,
    max_sentences=2500,
    update_every=8,
    rule_threshold=5e-3,
    seed=42,
):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()

    filtered_corpus = [s for s in train_corpus if 2 <= len(s) <= max_len]
    filtered_corpus.sort(key=len)
    print(f"Eligible sentences (len <= {max_len}): {len(filtered_corpus)}")

    if max_sentences and len(filtered_corpus) > max_sentences:
        rng = random.Random(seed)
        filtered_corpus = sorted(rng.sample(filtered_corpus, max_sentences), key=len)
        print(f"Training on sampled subset: {len(filtered_corpus)} sentences")
    else:
        print(f"Training on full filtered set: {len(filtered_corpus)} sentences")

    history = []
    if not filtered_corpus:
        return history

    for epoch in range(epochs):
        random.shuffle(filtered_corpus)
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        t0 = time.time()

        for idx, sentence in enumerate(filtered_corpus, start=1):
            loss = compute_inside_loss(model, sentence) / update_every
            loss.backward()
            total_loss += float(loss.item()) * update_every

            if idx % update_every == 0 or idx == len(filtered_corpus):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            if idx % 250 == 0:
                print(
                    f"  Epoch {epoch + 1} | Step {idx}/{len(filtered_corpus)} | "
                    f"Avg Loss: {total_loss / idx:.4f}"
                )

        avg_loss = total_loss / len(filtered_corpus)
        active_rules = count_active_rules(model, threshold=rule_threshold)
        dt = time.time() - t0
        history.append(
            {
                'epoch': epoch + 1,
                'avg_loss': avg_loss,
                'avg_ll': -avg_loss,
                'active_rules': active_rules,
                'sentences': len(filtered_corpus),
                'time': dt,
            }
        )

        print(
            f"=== Epoch {epoch + 1} Complete | Total Avg Loss: {avg_loss:.4f} | "
            f"Avg LL: {-avg_loss:.4f} | Rules > {rule_threshold:g}: {active_rules} | "
            f"{dt:.1f}s ==="
        )

    return history


In [6]:
def _select_binary_rules(rules, target_binary_rules=50, min_prob=1e-6, min_rules_per_lhs=2):
    candidates = {}
    for a in range(rules.shape[0]):
        current = []
        for b in range(rules.shape[1]):
            for c in range(rules.shape[2]):
                prob = float(rules[a, b, c])
                if prob > min_prob:
                    current.append((prob, a, b, c))
        current.sort(reverse=True, key=lambda x: x[0])
        if current:
            candidates[a] = current

    if not candidates:
        return []

    lhs_order = [0] + [
        lhs for lhs, _ in sorted(
            ((lhs, sum(prob for prob, _, _, _ in items)) for lhs, items in candidates.items()),
            key=lambda item: item[1],
            reverse=True,
        )
        if lhs != 0
    ]

    selected = []
    selected_keys = set()

    for round_idx in range(min_rules_per_lhs):
        for a in lhs_order:
            if len(selected) >= target_binary_rules:
                break
            if round_idx >= len(candidates.get(a, [])):
                continue
            prob, _, b, c = candidates[a][round_idx]
            key = (a, b, c)
            if key in selected_keys:
                continue
            selected.append((prob, a, b, c))
            selected_keys.add(key)
        if len(selected) >= target_binary_rules:
            break

    leftovers = []
    for a, items in candidates.items():
        for prob, _, b, c in items:
            key = (a, b, c)
            if key not in selected_keys:
                leftovers.append((prob, a, b, c))

    leftovers.sort(reverse=True, key=lambda x: x[0])
    for prob, a, b, c in leftovers:
        if len(selected) >= target_binary_rules:
            break
        key = (a, b, c)
        if key in selected_keys:
            continue
        selected.append((prob, a, b, c))
        selected_keys.add(key)

    return selected


def export_distilled_grammar(
    model,
    output_csv="pcfg3_modern_valid.csv",
    submission_csv="pcfg3_modern.csv",
    target_binary_rules=50,
    min_prob=1e-6,
    min_rules_per_lhs=2,
):
    if submission_csv and submission_csv == output_csv:
        raise ValueError("output_csv and submission_csv must be different files.")

    rules = model.get_dense_rules().detach().cpu().numpy()
    selected_rules = _select_binary_rules(
        rules,
        target_binary_rules=target_binary_rules,
        min_prob=min_prob,
        min_rules_per_lhs=min_rules_per_lhs,
    )

    lhs_groups = collections.defaultdict(list)
    for prob, a, b, c in selected_rules:
        lhs_groups[a].append((prob, b, c))

    rows = []
    rid = 1

    for i in range(N_PRETERM):
        rows.append(
            {
                'ID': rid,
                'LHS': sym_name(i),
                'LHS Type': 'preterminal',
                'RHS': chr(ord('a') + i),
                'Probability': 1.0,
            }
        )
        rid += 1

    for a in sorted(lhs_groups):
        children = lhs_groups[a]
        total_mass = sum(prob for prob, _, _ in children) or 1.0
        for prob, b, c in children:
            rows.append(
                {
                    'ID': rid,
                    'LHS': sym_name(a + N_PRETERM),
                    'LHS Type': 'nonterminal',
                    'RHS': f"{sym_name(b)} {sym_name(c)}",
                    'Probability': round(float(prob / total_mass), 6),
                }
            )
            rid += 1

    with open(output_csv, 'w', newline='') as f:
        writer = csv.DictWriter(
            f,
            fieldnames=['ID', 'LHS', 'LHS Type', 'RHS', 'Probability'],
        )
        writer.writeheader()
        writer.writerows(rows)

    if submission_csv:
        with open(submission_csv, 'w', newline='') as f:
            writer = csv.writer(f)
            for row in rows:
                writer.writerow([row['LHS'], row['RHS'], row['Probability']])

    nt_count = sum(1 for row in rows if row['LHS Type'] == 'nonterminal')
    print(f"Exported {len(rows)} rules ({nt_count} nonterminal) -> {output_csv}")
    if submission_csv:
        print(f"Submission-format copy -> {submission_csv}")

    if nt_count:
        print(f"\n{'=' * 60}")
        print("Discovered nonterminal expansion rules:")
        print(f"{'=' * 60}")
        for row in rows:
            if row['LHS Type'] == 'nonterminal':
                print(
                    f"  {row['LHS']:>6s} -> {row['RHS']:<16s}  "
                    f"P={row['Probability']:.4f}"
                )

    return rows


In [ ]:
if __name__ == "__main__":
    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    n_nt = 12
    rank = 24
    epochs = 6
    max_len = 40
    max_sentences = 2500
    update_every = 8
    rule_threshold = 5e-3

    try:
        train_corpus = load_corpus("sample/pcfg3_10k.txt", device=device)
    except FileNotFoundError:
        print("Warning: 'sample/pcfg3_10k.txt' not found. Using tiny dummy corpus for testing.")
        train_corpus = [
            torch.tensor([0, 1, 2], dtype=torch.long, device=device),
            torch.tensor([1, 2, 3, 4], dtype=torch.long, device=device),
        ]

    model = TD_PCFG(n_nt=n_nt, rank=rank).to(device)
    history = train_model(
        model,
        train_corpus,
        epochs=epochs,
        lr=0.02,
        max_len=max_len,
        max_sentences=max_sentences,
        update_every=update_every,
        rule_threshold=rule_threshold,
        seed=seed,
    )


In [ ]:
# 7. Final export
if 'model' not in globals():
    raise RuntimeError("Run the training cell first to create `model`.")

OUTPUT_CSV = "pcfg3_modern_valid.csv"
SUBMISSION_CSV = "pcfg3_modern.csv"
TARGET_BINARY_RULES = 50

rows = export_distilled_grammar(
    model,
    output_csv=OUTPUT_CSV,
    submission_csv=SUBMISSION_CSV,
    target_binary_rules=TARGET_BINARY_RULES,
    min_rules_per_lhs=2,
)


In [ ]:
# 8. Sanity check: log-likelihood and parse coverage

if 'model' not in globals() or 'train_corpus' not in globals():
    raise RuntimeError("Run the training cell first to create `model` and `train_corpus`.")
if 'rows' not in globals():
    raise RuntimeError("Run the export cell first to create `rows`.")

TEST_N = min(1000, len(train_corpus))

test_sents = train_corpus[-TEST_N:]
test_ll = 0.0
parsed = 0
failed = 0
fail_by_len = collections.Counter()

model.eval()
for sent in test_sents:
    n = len(sent)
    if n < 2:
        continue

    sp = compute_sentence_probability(model, sent)
    if sp > 1e-300:
        test_ll += math.log(sp)
        parsed += 1
    else:
        failed += 1
        fail_by_len[n] += 1

avg_ll = test_ll / max(parsed, 1)
coverage = parsed / max(parsed + failed, 1) * 100
rule_count = sum(1 for row in rows if row['LHS Type'] == 'nonterminal')

print(f"Evaluated {parsed + failed} sentences:")
print(f"  Parsed:   {parsed}  ({coverage:.1f}%)")
print(f"  Failed:   {failed}")
print(f"  Avg LL:   {avg_ll:.4f}")
print(f"  Rules:    {rule_count} nonterminal (target ~= {TARGET_BINARY_RULES})")

if failed > 0:
    print("\nFailed by sentence length (top 10):")
    for length, count in fail_by_len.most_common(10):
        print(f"  len={length:3d}: {count} failures")

if coverage < 70:
    print(f"\nWARNING: Low coverage ({coverage:.0f}%). Try:")
    print("  - Increasing n_nt (more nonterminals)")
    print("  - Training for more epochs")
    print("  - Increasing max_len")
elif coverage > 90:
    print("\nGood coverage! Check rule count and submit to Kaggle.")


In [ ]:
# 9. Plot training convergence
import matplotlib.pyplot as plt

if 'history' not in globals() or not history:
    raise RuntimeError("Run the training cell first to populate `history`.")

epochs = [h['epoch'] for h in history]
avg_lls = [h['avg_ll'] for h in history]
active_rules = [h['active_rules'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, avg_lls, 'b.-')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Avg Log-Likelihood')
ax1.set_title('Training Convergence (higher = better)')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, active_rules, 'r.-')
ax2.axhline(y=50, color='g', linestyle='--', alpha=0.5, label='target ~= 50')
ax2.set_xlabel('Epoch')
ax2.set_ylabel(f'# Rules > {rule_threshold:g}')
ax2.set_title('Active Rule Count Over Training')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
